# GenAI Pipeline — Testing Notebook

LLM-based screening of publications for alternative protein relevance and pillar classification.
Uses Claude (`claude-opus-4-8`) with prompt caching via the Anthropic Python SDK.

### 1. Imports and Configuration

In [ ]:
import duckdb
import pandas as pd
import json
import random
import time
import anthropic
import os
from pathlib import Path
from dotenv import load_dotenv
from pydantic import BaseModel, Field
from typing import Literal

load_dotenv()

DB_PATH = "../publications.db"
PROMPT_PATH = "prompt_unbiased_v1.md"
OUTPUT_DIR = Path(".")


### 2. Data Inspection

In [2]:
con = duckdb.connect(DB_PATH, read_only=True)
print("Tables:")
print(con.sql("SHOW TABLES").df()) # Check available tables

df = con.sql("SELECT * FROM publications_raw").df()
con.close()

print(f"\nShape: {df.shape}")
print(f"\nColumns: {list(df.columns)}")
print(f"\nScope distribution:")
print(df["scope"].value_counts())
print(f"\nPillar distribution:")
print(df["pillar"].value_counts())
df.head()

Tables:
               name
0  publications_raw

Shape: (4815, 7)

Columns: ['id', 'title', 'abstract', 'year', 'scope', 'pillar', 'research_category']

Scope distribution:
scope
out    3773
in     1042
Name: count, dtype: int64

Pillar distribution:
pillar
NA    3773
PB     695
F      151
CC      98
CM      98
Name: count, dtype: int64


,id,title,abstract,year,scope,pillar,research_category
0,pub.1192791591,Mushroom: an emerging source for next generati...,"Background: In recent years, plant-based and a...",2025,in,PB,Ingredient optimisation
1,pub.1187762379,Plant-Based Alternatives to Meat Products,Animal proteins have been used in the formulat...,2025,in,PB,Other
2,pub.1193391974,Influence of processing on protein quality and...,While meat is an established source of high-qu...,2025,in,PB,Ingredient optimisation
3,pub.1188899046,Solid State Fermentation—A Promising Approach ...,The increasing demand for sustainable dietary ...,2025,in,F,Other
4,pub.1193352421,"Physicochemical, Microbiological and Sensory E...",The bioactive properties of a phenolic extract...,2025,in,PB,End product formulation


### 3. Balanced Subset Creation

Two test sets with balanced representation across scope and pillar:
- `initial_test_data`: 14 records (6 out, 2 PB, 2 F, 2 cultivated, 2 cross-cutting)
- `test_data_100`: 110 records (40 out, 30 PB, 15 F, 15 cultivated, 10 cross-cutting)


In [9]:
def create_balanced_sample(df, n_out, n_pb, n_f, n_cultivated, n_cross, random_state=123): # Can change random_state for different samples - must be integer for reproducibility
    out_sample = df[df["scope"] == "out"].sample(n=n_out, random_state=random_state)
    pb_sample = df[df["pillar"] == "PB"].sample(n=n_pb, random_state=random_state)
    f_sample = df[df["pillar"] == "F"].sample(n=n_f, random_state=random_state)
    cult_sample = df[df["pillar"] == "CM"].sample(n=n_cultivated, random_state=random_state)
    cross_sample = df[df["pillar"] == "CC"].sample(n=n_cross, random_state=random_state)
    combined = pd.concat(
        [out_sample, pb_sample, f_sample, cult_sample, cross_sample], ignore_index=True
    )
    return combined.sample(frac=1, random_state=random_state).reset_index(drop=True)

In [10]:
initial_test_data = create_balanced_sample(df, n_out=6, n_pb=2, n_f=2, n_cultivated=2, n_cross=2)
print(f"initial_test_data: {initial_test_data.shape}")
# initial_test_data[["id", "title", "scope", "pillar"]]

initial_test_data: (14, 7)


In [11]:
PILLAR_ORDER = ["PB", "F", "CM", "CC", "NA"]

initial_test_data_review = initial_test_data.copy()
initial_test_data_review["pillar"] = pd.Categorical(
    initial_test_data_review["pillar"], categories=PILLAR_ORDER, ordered=True
)
initial_test_data_review.sort_values("pillar")[["id", "title", "scope", "pillar", "abstract"]]

,id,title,scope,pillar,abstract
0,pub.1191456334,"Comparative Analysis of Composition, Texture, ...",in,PB,The increasing demand for plant-based foods ha...
10,pub.1196236950,The influence of phytase on the solubility of ...,in,PB,Rapeseed (Brassica napus subsp. napus) protein...
5,pub.1185503350,From a Coriander Mayonnaise to a Vegan Analogu...,in,F,"History aside, traditional mayonnaise faces a ..."
6,pub.1192499872,Microalgal biomass in the European food indust...,in,F,The growing need for sustainable and nutrient-...
1,pub.1193443783,Proliferation and metabolic activity of the At...,in,CM,Introduction: Fish cell lines represent a powe...
7,pub.1192676912,Consumer Trust in Emerging Food Technologies: ...,in,CM,Consumer trust plays a critical role in the su...
11,pub.1196060665,Nutrient Equivalence of Plant-Based and Cultur...,in,CC,Meat provides high-quality protein and essenti...
13,pub.1193813897,The Potential of Fermentation-Based Processing...,in,CC,Proteins are fundamental to food systems due t...
2,pub.1184066957,Screening of Plant UDP-Glycosyltransferases fo...,out,NA,To cover the rising demand for natural food dy...
3,pub.1185271226,Construction of a CRISPR‐Cas9‐Based Genetic Ed...,out,NA,Putrescine plays a significant role in green f...


In [12]:
test_data_100 = create_balanced_sample(df, n_out=40, n_pb=30, n_f=15, n_cultivated=15, n_cross=10)
print(f"test_data_100: {test_data_100.shape}")
print(test_data_100[["scope", "pillar"]].value_counts())

test_data_100: (110, 7)
scope  pillar
out    NA        40
in     PB        30
       CM        15
       F         15
       CC        10
Name: count, dtype: int64


### 4. Save Subsets to Excel
Allows manual check of files selected. Consider whether those in the test sets are borderline cases or clear cut.

In [13]:
def save_subset(df, filename, output_dir=OUTPUT_DIR):
    path = output_dir / filename
    df.to_excel(path, index=False)
    print(f"Saved {len(df)} records to {path}")

save_subset(initial_test_data, "initial_test_data.xlsx")
save_subset(test_data_100, "test_data_100.xlsx")

Saved 14 records to initial_test_data.xlsx
Saved 110 records to test_data_100.xlsx


### 5. Load Prompt

In [ ]:
def load_prompt(path=PROMPT_PATH):
    with open(path, "r", encoding="utf-8") as f:
        prompt_text = f.read()
    return prompt_text.strip()

system_prompt = load_prompt()
print(system_prompt)


### 6. API Call with Prompt Caching

In [ ]:
# API config

# Anthropic model options — pricing as of 2026-06-10.
# Verify at https://www.anthropic.com/pricing if costs may have changed.
# Model                  Input $/1M   Output $/1M   Context
# claude-haiku-4-5         $1.00         $5.00       200K
# claude-sonnet-4-6        $3.00        $15.00       1M
# claude-opus-4-8          $5.00        $25.00       1M
MODELS = {
    "haiku":  "claude-haiku-4-5",
    "sonnet": "claude-sonnet-4-6",
    "opus":   "claude-opus-4-8",
}
MODEL = MODELS["haiku"]  # ← change this to switch model

MAX_TOKENS = 512         # max tokens in response; adjust based on expected reasoning length and cost tolerance
TEMPERATURE = 0.0        # 0.0 = deterministic; raise to ~0.3 to sample variance across REPETITIONS
CALL_DELAY = 1.0         # seconds between API calls
REQUEST_TIMEOUT = 120    # seconds before giving up on a single API call
MAX_RETRIES = 6          # retry attempts on rate-limit / transient errors
RETRY_BASE_SECONDS = 5.0  # exponential backoff base
RETRY_MAX_SECONDS = 90.0  # cap on backoff sleep

REPETITIONS = 1  # number of full runs; increase to measure output variance across runs

# ================================================================
# CHECKPOINT CONFIG
# Saves progress after each record; a run interrupted mid-way can
# be resumed without re-processing completed records.
# Set RESUME_INCOMPLETE = False to always start from scratch.
# ================================================================
CHECKPOINT_DIR = Path("checkpoints")
RESUME_INCOMPLETE = True

# ================================================================
# REASONING TOGGLE
# Keep True during testing — reasoning shows WHY the model decides
# as it does, which is essential for evaluating prompt quality.
# Set to False for production runs once the prompt is validated,
# to reduce token usage.
# ================================================================
INCLUDE_REASONING = True

class _ClassificationBase(BaseModel):
    in_scope: Literal["in", "out"]
    confidence: int = Field(ge=1, le=5)
    plant_based: bool
    fermentation: bool
    cultivated: bool

class _ClassificationWithReasoning(_ClassificationBase):
    reasoning: str

ClassificationSchema = _ClassificationWithReasoning if INCLUDE_REASONING else _ClassificationBase


client = anthropic.Anthropic(api_key=os.getenv("ANTHROPIC_API_KEY"))

def classify_publication(title, abstract, system_prompt):
    user_message = f"Title: {title}\n\nAbstract: {abstract}"
    response = client.messages.parse(
        model=MODEL,
        max_tokens=MAX_TOKENS,
        temperature=TEMPERATURE,
        timeout=REQUEST_TIMEOUT,
        system=[
            {
                "type": "text",
                "text": system_prompt,
                "cache_control": {"type": "ephemeral"}
            }
        ],
        messages=[
            {"role": "user", "content": user_message}
        ],
        output_format=ClassificationSchema,
    )
    return response.parsed_output


In [ ]:
def is_retryable_error(exc: Exception) -> bool:
    markers = ["503", "UNAVAILABLE", "RESOURCE_EXHAUSTED", "429",
               "TIMEOUT", "TIMED OUT", "READTIMEOUT", "CONNECTTIMEOUT"]
    return any(m in str(exc).upper() for m in markers)

def retry_sleep_seconds(attempt: int) -> float:
    sleep = min(RETRY_MAX_SECONDS, RETRY_BASE_SECONDS * (2 ** attempt))
    jitter = random.uniform(0.0, min(3.0, sleep * 0.2))
    return sleep + jitter


### 7. Error Handling with Retry

In [ ]:
def classify_with_error_handling(row, system_prompt):
    pub_id = row["id"]
    last_error = None
    for attempt in range(MAX_RETRIES + 1):
        try:
            result = classify_publication(row["title"], row["abstract"], system_prompt)
            if result is None:
                print(f"  Parse failed for {pub_id}: model returned no structured output")
                return {"id": pub_id, "status": "parse_error", "error": "no structured output"}
            output = result.model_dump()
            output["id"] = pub_id
            output["status"] = "ok"
            return output
        except anthropic.APIError as e:
            last_error = e
            if attempt >= MAX_RETRIES:
                break
            if is_retryable_error(e):
                sleep_s = retry_sleep_seconds(attempt)
                print(f"  Retryable error (attempt {attempt + 1}/{MAX_RETRIES}): {e}. Sleeping {sleep_s:.1f}s.")
                time.sleep(sleep_s)
            else:
                break
    print(f"  API error for {pub_id}: {last_error}")
    return {"id": pub_id, "status": "api_error", "error": str(last_error)}


### 8. Checkpoint Helpers

In [ ]:
def get_checkpoint_path(run_idx: int) -> Path:
    CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
    return CHECKPOINT_DIR / f"run_{run_idx}_checkpoint.json"

def save_checkpoint(run_idx: int, completed_results: list) -> None:
    path = get_checkpoint_path(run_idx)
    payload = {
        "run_idx": run_idx,
        "completed_ids": [r["id"] for r in completed_results],
        "results": completed_results,
    }
    path.write_text(json.dumps(payload, ensure_ascii=False), encoding="utf-8")

def load_checkpoint(run_idx: int):
    path = get_checkpoint_path(run_idx)
    if not path.exists():
        return None
    try:
        return json.loads(path.read_text(encoding="utf-8"))
    except Exception:
        return None

def delete_checkpoint(run_idx: int) -> None:
    path = get_checkpoint_path(run_idx)
    if path.exists():
        path.unlink()


### 9. Run on Initial Test Data

In [ ]:
# ← CHANGE THIS to switch between datasets
# Options: initial_test_data (14 records), test_data_100 (110 records)
DATASET = initial_test_data

all_results = []

for rep in range(REPETITIONS):
    run_idx = rep + 1
    print(f"\n{'='*50}\nRun {run_idx} / {REPETITIONS}\n{'='*50}")

    checkpoint = load_checkpoint(run_idx) if RESUME_INCOMPLETE else None
    if checkpoint:
        completed_results = checkpoint["results"]
        completed_ids = set(checkpoint["completed_ids"])
        print(f"  Resuming: {len(completed_ids)} records already processed.")
    else:
        completed_results, completed_ids = [], set()

    remaining = DATASET[~DATASET["id"].isin(completed_ids)]
    total = len(DATASET)

    for _, row in remaining.iterrows():
        n_done = len(completed_results)
        print(f"  [{n_done + 1}/{total}] {row['id']}")
        result = classify_with_error_handling(row, system_prompt)
        result["run"] = run_idx
        completed_results.append(result)
        save_checkpoint(run_idx, completed_results)
        if n_done + 1 < total:
            time.sleep(CALL_DELAY)

    delete_checkpoint(run_idx)
    all_results.extend(completed_results)

results_df = pd.DataFrame(all_results)
print(f"\nCompleted: {len(results_df)} records across {REPETITIONS} run(s)")
print(f"Successful: {(results_df['status'] == 'ok').sum()}")
print(f"Errors: {(results_df['status'] != 'ok').sum()}")
results_df


In [ ]:
# Compare LLM predictions against ground truth labels
result_cols = ["id", "run", "in_scope", "confidence", "plant_based", "fermentation", "cultivated", "status"]
if INCLUDE_REASONING:
    result_cols.insert(result_cols.index("confidence") + 1, "reasoning")

comparison = DATASET[["id", "scope", "pillar"]].merge(
    results_df[result_cols], on="id", how="left"
)
comparison["correct_scope"] = comparison["scope"] == comparison["in_scope"]
print(f"Overall scope accuracy ({REPETITIONS} run(s)): {comparison['correct_scope'].mean():.0%}")
if REPETITIONS > 1:
    per_run = comparison.groupby("run")["correct_scope"].mean().rename("scope_accuracy")
    print(f"\nPer-run accuracy:\n{per_run.to_string()}")
comparison
